# bronze_ingest

Respaldo para correr a mano la **Tarea 1** del DAG. La Parte A carga los Parquet que no pasan por el pipeline y la Parte B hace la ingesta con DLT (la misma que ejecuta Airflow).

Servicios: Spark `spark://spark-master:7077`, MinIO `http://minio:9000`, catálogo Nessie `http://nessie:19120`.

# Bronze - Parte A: carga manual
El enunciado dice que solo `users_2021` pasa por el pipeline con DLT. El resto de Bronze lo cargamos a mano
una sola vez: `post_2020`, `post_2021` y `users_2020`.

Los archivos originales son enormes, así que los recortamos a unos meses (por defecto enero a junio) con
DuckDB, que filtra Parquet grandes sin llenar la memoria, y después los subimos a MinIO (bucket `bronze`).
Quedan en: `bronze/post/2020/post_2020.parquet`, `bronze/post/2021/post_2021.parquet`
y `bronze/users/2020/users_2020.parquet`.

In [ ]:
import os
import duckdb
import s3fs

RAW_DIR = os.getenv("RAW_DIR", "/data/raw")
OUT_DIR = os.getenv("PREPARED_DIR", "/data/prepared")
MES_DESDE = int(os.getenv("FECHA_DESDE_MES", "1"))
MES_HASTA = int(os.getenv("FECHA_HASTA_MES", "6"))
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "http://minio:9000")
BUCKET = "bronze"

# (archivo original, nombre de la tabla en bronze, año)
CARGAS_MANUALES = [
    ("posts_2020.parquet", "post", 2020),
    ("posts_2021.parquet", "post", 2021),
    ("users.parquet", "users", 2020),   # users.parquet trae todos los años, acá nos quedamos con los de 2020
]

In [ ]:
def rango_fechas(anio, mes_desde, mes_hasta):
    """Arma el rango de fechas [inicio, fin) de los meses que queremos, como texto en UTC para usarlo en el SQL."""
    fin_anio, fin_mes = (anio + 1, 1) if mes_hasta == 12 else (anio, mes_hasta + 1)
    return f"{anio}-{mes_desde:02d}-01 00:00:00+00", f"{fin_anio}-{fin_mes:02d}-01 00:00:00+00"


def recortar_parquet(origen, destino, anio, mes_desde, mes_hasta):
    """Lee el Parquet original y guarda solo las filas creadas en ese año y esos meses. Devuelve cuántas filas quedaron."""
    inicio, fin = rango_fechas(anio, mes_desde, mes_hasta)
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    con = duckdb.connect()
    con.execute("SET memory_limit='1GB'; SET threads=2; SET temp_directory='/tmp/duckdb_tmp'")
    con.execute(f"""
        COPY (
            SELECT * FROM read_parquet('{origen}')
            WHERE CreationDate >= TIMESTAMPTZ '{inicio}' AND CreationDate < TIMESTAMPTZ '{fin}'
        ) TO '{destino}' (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)
    filas = con.execute(f"SELECT count(*) FROM read_parquet('{destino}')").fetchone()[0]
    con.close()
    return filas


def subir_a_minio(ruta_local, clave):
    """Sube el archivo al bucket bronze. Si ya existía lo reemplaza, que es lo que queremos en Bronze (override)."""
    fs = s3fs.S3FileSystem(
        key=os.getenv("AWS_ACCESS_KEY_ID", "admin"),
        secret=os.getenv("AWS_SECRET_ACCESS_KEY", "password"),
        client_kwargs={"endpoint_url": S3_ENDPOINT, "region_name": "us-east-1"},
    )
    fs.put(ruta_local, f"{BUCKET}/{clave}")

In [ ]:
for archivo, tabla, anio in CARGAS_MANUALES:
    origen = os.path.join(RAW_DIR, archivo)
    destino = os.path.join(OUT_DIR, f"{tabla}_{anio}.parquet")
    clave = f"{tabla}/{anio}/{tabla}_{anio}.parquet"
    filas = recortar_parquet(origen, destino, anio, MES_DESDE, MES_HASTA)
    subir_a_minio(destino, clave)
    print(f"[bronze manual] s3://{BUCKET}/{clave}: {filas:,} filas (meses {MES_DESDE}-{MES_HASTA} de {anio})")

# Bronze - Parte B: ingesta con DLT (la que corre Airflow)
Esta es la Tarea 1 del DAG. DLT lee la fuente `users` (el Parquet de StackOverflow), se queda con los
usuarios del año que le indiquemos (por defecto 2021, de enero a junio) y los guarda en el bucket
`bronze` de MinIO como Parquet: `bronze/users/2021/users_2021.parquet`.

La escritura es **override**: usamos `write_disposition="replace"` y un nombre de archivo fijo, así que
si corremos el pipeline varias veces siempre se sobrescribe el mismo archivo y no se duplica nada.

In [ ]:
import os
import datetime as dt
import dlt
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# En Bronze dejamos los nombres de las columnas tal cual vienen de la fuente (son datos crudos)
os.environ.setdefault("SCHEMA__NAMING", "direct")

ORIGEN = os.getenv("USERS_SOURCE", "/data/raw/users.parquet")
ANIO = int(os.getenv("USERS_YEAR", "2021"))
MES_DESDE = int(os.getenv("FECHA_DESDE_MES", "1"))
MES_HASTA = int(os.getenv("FECHA_HASTA_MES", "6"))

In [ ]:
def rango_fechas(anio, mes_desde, mes_hasta):
    utc = dt.timezone.utc
    inicio = dt.datetime(anio, mes_desde, 1, tzinfo=utc)
    fin = dt.datetime(anio + 1, 1, 1, tzinfo=utc) if mes_hasta == 12 else dt.datetime(anio, mes_hasta + 1, 1, tzinfo=utc)
    return inicio, fin


@dlt.resource(name="users", write_disposition="replace")
def users():
    """Lee el Parquet original por lotes (para no llenar la memoria) y entrega solo los usuarios creados en esos meses."""
    inicio, fin = rango_fechas(ANIO, MES_DESDE, MES_HASTA)
    tipo = pa.timestamp("ms", tz="UTC")
    ini, fi = pa.scalar(inicio, type=tipo), pa.scalar(fin, type=tipo)
    pf = pq.ParquetFile(ORIGEN)
    for lote in pf.iter_batches(batch_size=250_000):
        t = pa.Table.from_batches([lote])
        sub = t.filter(pc.and_(pc.greater_equal(t["CreationDate"], ini), pc.less(t["CreationDate"], fi)))
        if sub.num_rows:
            yield sub

In [ ]:
# DLT guarda los archivos en <bucket>/<dataset_name>/<layout>. Con dataset_name="users" y este patrón
# el archivo queda en bronze/users/2021/users_2021.parquet, siempre con el mismo nombre (así se sobrescribe).
destino = dlt.destinations.filesystem(
    layout=str(ANIO) + "/{table_name}_" + str(ANIO) + ".{ext}",
    warn_unsafe_layout_separators=False,
)
pipeline = dlt.pipeline(
    pipeline_name="bronze_users",
    destination=destino,
    dataset_name="users",
    pipelines_dir=os.getenv("DLT_PIPELINES_DIR", "/tmp/dlt_pipelines"),
)
info = pipeline.run(users(), loader_file_format="parquet")
print(info)
print(f"[bronze dlt] users {ANIO} (meses {MES_DESDE}-{MES_HASTA}) cargado en s3://bronze/users/{ANIO}/users_{ANIO}.parquet")